# Train Qwen LoRA trên 24 business — giữ riêng 6 business

1. Trong Colab A100 hiện tại, chọn **Runtime → Restart session** để giải phóng model inference cũ (giữ nguyên các file trên VM).
2. Mở notebook này, chạy các cell theo thứ tự. Cell upload chọn `zaic-business24-training.zip` từ máy local.
3. Cell cuối in adapter mới, không ghi đè model cũ. Tải bản backup adapter sau khi train xong trước khi đóng runtime.

633 mẫu từ train partition; 6 business validation không dùng SFT hay chọn epoch. Tất cả mẫu dài hơn 2.048 tokens; max 5.292. Script dùng trần 8.192 và dừng nếu bất kỳ mẫu nào bị vượt trần, không cắt âm thầm. Loss chỉ tính phần completion theo [TRL 0.25](https://huggingface.co/docs/trl/v0.25.0/en/sft_trainer).

Dữ liệu cũ và rule/prompt từng phân tích toàn Train: đây là đối chứng về ranh giới SFT, chưa phải đánh giá hoàn toàn unseen. Chưa có kết quả GPU của notebook này cho đến khi chạy thực tế.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, zipfile
uploaded = files.upload()  # Chọn zaic-business24-training.zip
archive = Path('/content/zaic-business24-training.zip')
assert archive.exists(), 'Chọn đúng file zaic-business24-training.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '80f4dc7b46070d2d3475a9a9642e253c4e11d0977a6068ccd1c6110d2275d90c', 'Bundle SHA256 không khớp'
work = Path('/content/zaic-holdout')
work.mkdir(exist_ok=True)
expected = {'holdout_training.py', 'split.json', 'requirements.txt', 'train-only-sft.manifest.json', 'train-only-sft.jsonl'}
with zipfile.ZipFile(archive) as z:
    assert set(z.namelist()) == expected
    for name in expected:
        (work / name).write_bytes(z.read(name))
print('Bundle verified:', work)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '/content/zaic-holdout/requirements.txt'], check=True)


In [ ]:
import subprocess, sys
TRAIN = [sys.executable, '/content/zaic-holdout/holdout_training.py',
         '--revision', 'c03e6d358207e414f1eca0bb1891e29f1db0e242']
subprocess.run(TRAIN + ['--preflight-only'], check=True)


In [ ]:
# Train từ Qwen gốc (không tiếp tục checkpoint cũ).
# Nếu VRAM không đủ: Runtime → Restart session, rồi chạy lại các cell.
subprocess.run(TRAIN, check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, json
adapter = Path('/content/qwen2.5-coder-7b-lora-business24')
manifest = json.loads((adapter/'training_manifest.json').read_text())
assert manifest['training_complete'] is True and not manifest['overlap_businesses']
print('ADAPTER:', adapter)
print('BASE REVISION:', manifest['base_revision'])
print('DATA SHA256:', manifest['data_sha256'])
print('Gửi training_manifest.json cho agent để xác nhận trước khi đo trên 6 business.')
files.download(str(adapter/'training_manifest.json'))
# Backup toàn adapter + provenance (không bao gồm intermediate checkpoint-*).
backup = Path('/content/business24-adapter-backup')
backup.mkdir(exist_ok=True)
for path in adapter.iterdir():
    if path.is_file():
        shutil.copy2(path, backup/path.name)
archive = shutil.make_archive('/content/business24-adapter-backup','zip',backup)
files.download(archive)


## Sau khi train

Giữ nguyên checkpoint cũ. Adapter mới ở `/content/qwen2.5-coder-7b-lora-business24`.

Sau khi agent kiểm tra `training_manifest.json`, bước tiếp theo là gộp adapter vào đúng Qwen revision hoặc phục vụ trực tiếp bằng PEFT, mở endpoint mới và chạy `evaluate_selective.py --training-data artifacts/train-only-sft.jsonl`. Không dùng endpoint cũ để đo checkpoint mới.
